# Image Schema Classification — BERT (English) + Symbolic Comparison

This notebook:
1. Trains a **BERT** (`bert-base-uncased`) classifier on **English-only** image schema data
2. Evaluates the symbolic (Framester/FrameNet-based) approach on a 100-sentence evaluation set
3. Compares both approaches side-by-side

In [ ]:
random_split_state = 44

In [ ]:
!pip install transformers
!pip install lime

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from sklearn import preprocessing
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import time
import datetime
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lime
from lime import lime_text

## 1. Data Exploration (English only)

In [ ]:
df_all = pd.read_csv("Image_Schemas_English_and_German.csv")
df_all.rename(columns={'IMAGE_SCHEMA_ANNOTATION': 'SuperordinateSchema'}, inplace=True)

# Keep English only
df = df_all.loc[df_all['Language'] == 'en'].copy()
print(f"Total English samples: {len(df)}")

In [ ]:
df.drop_duplicates(subset='LinguisticExamples', keep='first', inplace=True)
print(f"After deduplication: {len(df)} samples")

In [ ]:
df.SuperordinateSchema.value_counts()

### Top words per class

In [ ]:
labels_all = list(df.SuperordinateSchema.unique())
is_texts = [''] * len(labels_all)
for i, row in df.iterrows():
    ind = labels_all.index(row['SuperordinateSchema'])
    is_texts[ind] += row['LinguisticExamples']

vectorizer = CountVectorizer()
count_matrix = vectorizer.fit_transform(is_texts)
counts = pd.DataFrame(count_matrix.toarray(), index=labels_all, columns=vectorizer.get_feature_names_out())

# Show top 10 words for CONTAINMENT
counts.T.sort_values(by='CONTAINMENT', ascending=False).head(10)

## 2. Prepare Data for BERT Classifier

In [ ]:
# Remove less-studied schemas
schemas_to_remove = ["LINK", "OBJECT", "SUBSTANCE", "SPLITTING", "SUPPORT", "COVERING"]
for s in schemas_to_remove:
    df = df[df.SuperordinateSchema != s]

df["SuperordinateSchema"].value_counts().plot(
    kind='bar', title='Class Counts (English)', ylabel='Count', xlabel='Class', figsize=(8, 4)
)
plt.tight_layout()
plt.show()

In [ ]:
df["SuperordinateSchema"].value_counts()

In [ ]:
# Encode labels
labels = list(df.SuperordinateSchema.unique())
le = preprocessing.LabelEncoder()
le.fit(labels)
df["SuperordinateSchema"] = le.transform(df["SuperordinateSchema"])
labels = le.classes_
print("Classes:", labels)

In [ ]:
# Train/test split
train_data, test_data, y_train, y_test = train_test_split(
    df, df["SuperordinateSchema"],
    stratify=df["SuperordinateSchema"],
    test_size=0.2,
    random_state=random_split_state
)
print(f"Train: {len(train_data)} | Test: {len(test_data)}")

### Tokenization with BERT

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def tokenize_bert(data, max_len):
    input_ids_, attn_masks_, labels_ = [], [], []
    for _, row in data.iterrows():
        encoded = bert_tokenizer.encode_plus(
            row['LinguisticExamples'],
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids_.append(encoded['input_ids'])
        attn_masks_.append(encoded['attention_mask'])
        labels_.append(row['SuperordinateSchema'])
    return (
        torch.cat(input_ids_, dim=0),
        torch.cat(attn_masks_, dim=0),
        torch.tensor(labels_)
    )

In [ ]:
max_len = 128
input_ids_train, attn_masks_train, labels_train = tokenize_bert(train_data, max_len)
input_ids_test, attn_masks_test, labels_test   = tokenize_bert(test_data, max_len)
print("Tokenization done.")

In [ ]:
batch_size = 16
train_dataset = TensorDataset(input_ids_train, attn_masks_train, labels_train)
test_dataset  = TensorDataset(input_ids_test,  attn_masks_test,  labels_test)

train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset),     batch_size=batch_size)
test_dataloader  = DataLoader(test_dataset,  sampler=SequentialSampler(test_dataset),  batch_size=batch_size)

## 3. BERT Training

In [ ]:
def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round(elapsed))))

def softmax(z):
    s = np.max(z, axis=1, keepdims=True)
    e_x = np.exp(z - s)
    return e_x / np.sum(e_x, axis=1, keepdims=True)

In [ ]:
def validate(dataloader, model, verbose=True, print_cm=False, normalized=False):
    model.eval()
    total_loss = 0
    predictions, true_labels = [], []

    for batch in dataloader:
        b_ids, b_mask, b_labels = [x.to(device) for x in batch]
        with torch.no_grad():
            output = model(b_ids, token_type_ids=None, attention_mask=b_mask, labels=b_labels)
        total_loss += output.loss.item()
        predictions.append(output.logits.detach().cpu().numpy())
        true_labels.append(b_labels.cpu().numpy())

    flat_preds  = np.argmax(np.concatenate(predictions), axis=1)
    flat_true   = np.concatenate(true_labels)

    if verbose:
        print(classification_report(flat_true, flat_preds, target_names=labels))

    if print_cm:
        cm = confusion_matrix(flat_true, flat_preds, labels=list(range(len(labels))))
        data = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] if normalized else cm
        fmt  = '.2f' if normalized else 'd'
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(np.round(data, 2) if normalized else data,
                    annot=True, fmt=fmt, linewidths=.5,
                    xticklabels=labels, yticklabels=labels, ax=ax)
        plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()

    acc      = (flat_preds == flat_true).mean()
    _, _, mF1, _ = precision_recall_fscore_support(flat_true, flat_preds, average='macro', zero_division=0)
    wF1      = f1_score(flat_true, flat_preds, average='weighted', zero_division=0)
    avg_loss = total_loss / len(dataloader)
    return avg_loss, acc, mF1, wF1

In [ ]:
def train_model(epochs, model, train_dl, val_dl, optimizer, scheduler, random_seed=42, verbose=True):
    random.seed(random_seed); np.random.seed(random_seed)
    torch.manual_seed(random_seed); torch.cuda.manual_seed_all(random_seed)

    stats = []
    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        model.train()
        total_train_loss = 0
        t0 = time.time()

        for batch in train_dl:
            b_ids, b_mask, b_labels = [x.to(device) for x in batch]
            model.zero_grad()
            output = model(b_ids, token_type_ids=None, attention_mask=b_mask, labels=b_labels)
            output.loss.backward()
            total_train_loss += output.loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()

        avg_train = total_train_loss / len(train_dl)
        print(f"  Train loss: {avg_train:.4f}  ({format_time(time.time()-t0)})")

        print_cm = (epoch == epochs - 1)
        avg_val, acc, mF1, wF1 = validate(val_dl, model, verbose, print_cm)
        print(f"  Val loss: {avg_val:.4f} | Acc: {acc:.4f} | Macro F1: {mF1:.4f} | Weighted F1: {wF1:.4f}")

        stats.append({
            'epoch': epoch+1, 'Training Loss': avg_train, 'Valid. Loss': avg_val,
            'Accuracy': acc, 'Macro F1': mF1, 'Weighted F1': wF1
        })
    return stats

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

lr     = 3e-5
eps    = 1e-8
epochs = 12

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(labels))
bert_model.to(device)

optimizer = AdamW(bert_model.parameters(), lr=lr, eps=eps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_dataloader) * epochs
)

In [ ]:
training_stats = train_model(
    epochs=epochs,
    model=bert_model,
    train_dl=train_dataloader,
    val_dl=test_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    verbose=True
)

In [ ]:
# Save / Load model
SAVE_PATH = 'saved_models/bert_english_iscat'
bert_model.save_pretrained(SAVE_PATH)
print("Model saved to", SAVE_PATH)

In [ ]:
# Load saved model (run this cell to skip re-training)
# bert_model = BertForSequenceClassification.from_pretrained(SAVE_PATH)
# bert_model.to(device)

## 4. Training Curves

In [ ]:
df_stats = pd.DataFrame(training_stats).set_index('epoch')
df_stats

In [ ]:
sns.set(style='darkgrid')
plt.figure(figsize=(12, 5))
plt.plot(df_stats['Training Loss'], 'b-o', label='Training Loss')
plt.plot(df_stats['Valid. Loss'],   'g-o', label='Validation Loss')
plt.plot(df_stats['Weighted F1'],   'r-o', label='Weighted F1')
plt.title('BERT — Training & Validation')
plt.xlabel('Epoch'); plt.ylabel('Value')
plt.legend(); plt.tight_layout(); plt.show()

## 5. LIME Explanations (BERT)

In [ ]:
def bert_predict(sentences):
    """Wrapper for LIME: returns softmax probabilities."""
    predictions = np.empty((len(sentences), len(labels)))
    bert_model.eval()
    for i, sentence in enumerate(sentences):
        encoded = bert_tokenizer.encode_plus(
            sentence, max_length=max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        with torch.no_grad():
            output = bert_model(
                encoded['input_ids'].to(device),
                attention_mask=encoded['attention_mask'].to(device),
                labels=torch.tensor(0).to(device)
            )
        predictions[i] = softmax(output.logits.cpu().numpy())
    return predictions

explainer = lime_text.LimeTextExplainer(class_names=labels, verbose=False)

In [ ]:
# Global LIME stats over all English test samples
results = [{} for _ in labels]

for _, row in test_data.iterrows():
    exp = explainer.explain_instance(
        row['LinguisticExamples'], bert_predict,
        num_features=20, top_labels=len(labels), num_samples=500
    )
    pred_index = exp.top_labels[0]
    for word, importance in exp.as_list(label=pred_index):
        results[pred_index].setdefault(word, []).append(importance)

results_avg = [
    {w: sum(v)/len(v) for w, v in d.items()} for d in results
]

In [ ]:
# Plot top influential words per class
for i, label_name in enumerate(labels):
    res = sorted(results_avg[i].items(), key=lambda x: x[1], reverse=True)[:15]
    res.reverse()
    if not res:
        continue
    plt.figure(figsize=(9, 5))
    plt.barh([w for w, _ in res], [s for _, s in res])
    plt.title(f'LIME — {label_name}')
    plt.tight_layout(); plt.show()

## 6. Symbolic vs BERT — Comparison on the 100-sentence Evaluation Set

The file `100_for_eval_fnroles_out.csv` contains:
- `tweet_text` : the sentence
- `label` : gold-standard image schema
- `pred` : prediction(s) from the **symbolic** (Framester/FrameNet) approach
- `trig` : URIs that triggered the symbolic prediction

We run BERT on the same sentences to obtain a direct comparison.

In [ ]:
eval_df = pd.read_csv('100_for_eval_fnroles_out.csv')
eval_df.head()

### 6.1 Symbolic approach — extract first predicted label

In [ ]:
# Map symbolic schema names to the same format used by BERT labels
SCHEMA_MAP = {
    'center_periphery': 'CENTER_PERIPHERY',
    'containment':      'CONTAINMENT',
    'part_whole':       'PART_WHOLE',
    'source_path_goal': 'SOURCE_PATH_GOAL',
    'source':           'SOURCE_PATH_GOAL',
    'path':             'SOURCE_PATH_GOAL',
    'goal':             'SOURCE_PATH_GOAL',
    'support':          'SUPPORT',
    'blockage':         'BLOCKAGE',
    'container':        'CONTAINMENT',
    'part':             'PART_WHOLE',
    'whole':            'PART_WHOLE',
    'periphery':        'CENTER_PERIPHERY',
    'center':           'CENTER_PERIPHERY',
}

def normalize_schema(name):
    if not isinstance(name, str):
        return None
    name = name.strip().lower()
    return SCHEMA_MAP.get(name, name.upper())

# The symbolic pred column may contain multiple labels (comma-separated); keep the first
eval_df['symbolic_pred'] = eval_df['pred'].apply(
    lambda x: normalize_schema(str(x).split(',')[0]) if isinstance(x, str) else None
)

# Normalize gold labels too
eval_df['gold'] = eval_df['label'].apply(lambda x: normalize_schema(x))

eval_df[['tweet_text', 'gold', 'symbolic_pred']].head(10)

### 6.2 BERT predictions on the evaluation set

In [ ]:
def bert_predict_single(sentence):
    bert_model.eval()
    encoded = bert_tokenizer.encode_plus(
        sentence, max_length=max_len, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        output = bert_model(
            encoded['input_ids'].to(device),
            attention_mask=encoded['attention_mask'].to(device),
            labels=torch.tensor(0).to(device)
        )
    return labels[np.argmax(output.logits.cpu().numpy())]

eval_df['bert_pred'] = eval_df['tweet_text'].apply(bert_predict_single)
eval_df[['tweet_text', 'gold', 'symbolic_pred', 'bert_pred']].head(10)

### 6.3 Quantitative Comparison

In [ ]:
# Only keep rows where gold label is known
eval_known = eval_df[eval_df['gold'].notna()].copy()

gold = eval_known['gold']

# Symbolic: only rows where it produced a prediction
sym_mask = eval_known['symbolic_pred'].notna()
sym_gold = gold[sym_mask]
sym_pred = eval_known['symbolic_pred'][sym_mask]

# BERT predictions (always available)
bert_pred_col = eval_known['bert_pred']

print("=== Symbolic Approach ===")
print(f"Coverage: {sym_mask.sum()}/{len(eval_known)} sentences ({100*sym_mask.mean():.1f}%)")
print(classification_report(sym_gold, sym_pred, zero_division=0))

print("\n=== BERT (bert-base-uncased) ===")
print(f"Coverage: {len(eval_known)}/{len(eval_known)} sentences (100%)")
print(classification_report(gold, bert_pred_col, zero_division=0))

In [ ]:
# Summary table
def summary_metrics(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    _, _, mF1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro',    zero_division=0)
    wF1          = f1_score(y_true, y_pred,          average='weighted', zero_division=0)
    return {'System': name, 'Accuracy': round(acc,3), 'Macro F1': round(mF1,3), 'Weighted F1': round(wF1,3)}

rows = [
    summary_metrics(sym_gold,  sym_pred,      f'Symbolic (coverage {sym_mask.mean()*100:.0f}%)'),
    summary_metrics(gold,      bert_pred_col, 'BERT (bert-base-uncased)'),
]
comparison_df = pd.DataFrame(rows).set_index('System')
comparison_df

### 6.4 Confusion Matrices

In [ ]:
schema_classes = sorted(gold.unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (y_t, y_p, title) in zip(axes, [
    (sym_gold, sym_pred, 'Symbolic Approach'),
    (gold, bert_pred_col, 'BERT (bert-base-uncased)')
]):
    cm = confusion_matrix(y_t, y_p, labels=schema_classes)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(np.round(cm_norm, 2), annot=True, fmt='.2f', linewidths=.5,
                xticklabels=schema_classes, yticklabels=schema_classes, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Symbolic vs BERT (normalised)', fontsize=14)
plt.tight_layout(); plt.show()

### 6.5 Per-class F1 comparison (bar chart)

In [ ]:
from sklearn.metrics import f1_score

# Per-class F1 for each system
sym_f1  = f1_score(sym_gold,  sym_pred,      labels=schema_classes, average=None, zero_division=0)
bert_f1 = f1_score(gold,      bert_pred_col, labels=schema_classes, average=None, zero_division=0)

x = np.arange(len(schema_classes))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, sym_f1,  width, label='Symbolic')
ax.bar(x + width/2, bert_f1, width, label='BERT')
ax.set_xticks(x); ax.set_xticklabels(schema_classes, rotation=30, ha='right')
ax.set_ylabel('F1 Score'); ax.set_title('Per-class F1: Symbolic vs BERT')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

### 6.6 Error Analysis — sentences where both systems disagree with gold

In [ ]:
both_wrong = eval_known[
    (eval_known['bert_pred'] != eval_known['gold']) &
    (eval_known['symbolic_pred'] != eval_known['gold'])
][['tweet_text', 'gold', 'symbolic_pred', 'bert_pred']]

print(f"Sentences where BOTH systems are wrong: {len(both_wrong)}")
both_wrong

In [ ]:
# Sentences where BERT is correct but symbolic is wrong
bert_wins = eval_known[
    (eval_known['bert_pred'] == eval_known['gold']) &
    (eval_known['symbolic_pred'] != eval_known['gold'])
][['tweet_text', 'gold', 'symbolic_pred', 'bert_pred']]

print(f"BERT correct, Symbolic wrong: {len(bert_wins)}")
bert_wins.head(10)

In [ ]:
# Sentences where Symbolic is correct but BERT is wrong
sym_wins = eval_known[
    (eval_known['symbolic_pred'] == eval_known['gold']) &
    (eval_known['bert_pred'] != eval_known['gold'])
][['tweet_text', 'gold', 'symbolic_pred', 'bert_pred']]

print(f"Symbolic correct, BERT wrong: {len(sym_wins)}")
sym_wins.head(10)